# Branch 1 — Numerical Models — v4 Paper-Ready


### v4 paper-ready corrections

- same matched company-quarter observations and chronological splits;
- feature eligibility determined from TRAIN only;
- validation-only hyperparameter and threshold selection;
- repaired numerical feature ablation;
- information-availability audit;
- chronology-preserving optional leave-one-company-out robustness;
- paper-ready outputs with row identifiers and selected settings.


In [ ]:
# SIC 3674 multimodal dataset input — Kaggle + Colab aware
from pathlib import Path

TARGET_DATASET_NAMES = [
    "sic3674_multimodal_model_rows.parquet",
    "sic3674_multimodal_model_rows.csv",
]

def resolve_sic3674_dataset_path():
    # Kaggle first.
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            for filename in TARGET_DATASET_NAMES:
                matches = list(root.rglob(filename))
                if matches:
                    return matches[0]

    # Colab / Drive fallback.
    candidates = [
        Path("/content/drive/MyDrive/sic3674_output/sic3674_multimodal_model_rows.parquet"),
        Path("/content/drive/MyDrive/sic3674_output/sic3674_multimodal_model_rows.csv"),
        Path("/content/sic3674_output/sic3674_multimodal_model_rows.parquet"),
        Path("/content/sic3674_output/sic3674_multimodal_model_rows.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate sic3674_multimodal_model_rows.parquet/csv. "
        "Attach the dataset to Kaggle or place it in sic3674_output."
    )

SIC3674_DATA_PATH = resolve_sic3674_dataset_path()
print("Using SIC 3674 dataset:", SIC3674_DATA_PATH)


In [ ]:
!pip -q install pyarrow openpyxl xgboost sentence-transformers transformers accelerate beautifulsoup4 requests tqdm

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, log_loss, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH: Path | None = None
NEUTRAL_BAND = 0.02
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
THRESHOLD_GRID = np.linspace(0.20, 0.80, 121)

OUTPUT_DIR = Path('/content/semiconductor_branch_numerical')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load the same dataset used by v6.1

In [ ]:
def discover_data_path() -> Path:
    if "SIC3674_DATA_PATH" in globals() and Path(SIC3674_DATA_PATH).exists():
        return Path(SIC3674_DATA_PATH)

    candidates = [
        Path('/content/drive/MyDrive/sec_research_semiconductor/semiconductor_sec_numeric_text.parquet'),
        Path('/content/drive/MyDrive/sec_research_semiconductor/sec_experiment_semiconductor.parquet'),
        Path('/content/semiconductor_sec_numeric_text.parquet'),
        Path('/content/sec_experiment_semiconductor.parquet'),
        Path('semiconductor_sec_numeric_text.parquet'),
        Path('semiconductor_sec_numeric_text.csv'),
        Path('sec_experiment_semiconductor.parquet'),
        Path('model_dataset.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Dataset not found. Attach the SIC3674 multimodal dataset or set DATA_PATH."
    )

resolved_path = DATA_PATH if DATA_PATH is not None else discover_data_path()

if resolved_path.suffix.lower() == '.parquet':
    raw = pd.read_parquet(resolved_path)
elif resolved_path.suffix.lower() == '.csv':
    raw = pd.read_csv(resolved_path)
else:
    raise ValueError('Use a .parquet or .csv dataset.')

print('Loaded:', resolved_path)
print('Rows:', len(raw), '| Columns:', len(raw.columns))


## Rebuild target exactly as v6.1

In [ ]:
data = raw.copy()

# Standardize identifiers and dates.
if "cik" not in data.columns:
    if "ticker" not in data.columns:
        raise ValueError("The dataset must contain either cik or ticker.")
    data["cik"] = data["ticker"].astype(str)

if "ticker" not in data.columns:
    data["ticker"] = data["cik"].astype(str)

date_candidates = [
    "quarter_end",
    "feature_cutoff_date",
    "label_available_date",
]
for column in date_candidates:
    if column in data.columns:
        data[column] = pd.to_datetime(data[column], errors="coerce")

if "quarter_end" not in data.columns:
    raise ValueError("The dataset must contain quarter_end.")

data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

# Map names from the earlier proof-of-concept dataset when needed.
rename_aliases = {
    "revenue_mm": "revenue",
    "inventory_mm": "inventory",
    "accounts_receivable_mm": "accounts_receivable",
    "operating_cash_flow_mm": "operating_cash_flow",
    "inventory_ratio": "inventory_to_ttm_revenue",
    "ar_ratio": "receivables_to_ttm_revenue",
    "ocf_margin": "cash_flow_margin",
}
for old_name, new_name in rename_aliases.items():
    if new_name not in data.columns and old_name in data.columns:
        data[new_name] = pd.to_numeric(data[old_name], errors="coerce")

numeric_candidates = [
    "revenue",
    "revenue_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "gross_margin",
    "operating_margin",
    "cash_flow_margin",
    "inventory",
    "inventory_to_ttm_revenue",
    "accounts_receivable",
    "receivables_to_ttm_revenue",
    "capital_expenditures",
    "capex_to_revenue",
    "total_assets",
    "log_assets",
    "liabilities_to_assets",
]
for column in numeric_candidates:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Reconstruct YoY growth if it is missing and revenue is available.
if "revenue_yoy_growth" not in data.columns and "revenue" in data.columns:
    revenue_lag4 = grouped["revenue"].shift(4)
    quarter_lag4 = grouped["quarter_end"].shift(4)
    gap4 = (data["quarter_end"] - quarter_lag4).dt.days
    data["revenue_yoy_growth"] = np.where(
        gap4.between(320, 410),
        data["revenue"] / revenue_lag4 - 1.0,
        np.nan,
    )

# Reconstruct current momentum if missing.
if "revenue_momentum" not in data.columns:
    previous_growth = grouped["revenue_yoy_growth"].shift(1)
    previous_end = grouped["quarter_end"].shift(1)
    gap1 = (data["quarter_end"] - previous_end).dt.days
    data["revenue_momentum"] = np.where(
        gap1.between(60, 125),
        data["revenue_yoy_growth"] - previous_growth,
        np.nan,
    )

# Reconstruct next-quarter growth if missing.
if "next_quarter_growth" not in data.columns:
    next_growth = grouped["revenue_yoy_growth"].shift(-1)
    next_end = grouped["quarter_end"].shift(-1)
    next_gap = (next_end - data["quarter_end"]).dt.days
    data["next_quarter_growth"] = np.where(
        next_gap.between(60, 125),
        next_growth,
        np.nan,
    )

data["future_growth_change"] = (
    data["next_quarter_growth"] - data["revenue_yoy_growth"]
)

data["target_clean"] = pd.Series(
    np.select(
        [
            data["future_growth_change"] > NEUTRAL_BAND,
            data["future_growth_change"] < -NEUTRAL_BAND,
        ],
        [1.0, 0.0],
        default=np.nan,
    ),
    index=data.index,
)

data["target_status"] = np.select(
    [
        data["future_growth_change"] > NEUTRAL_BAND,
        data["future_growth_change"] < -NEUTRAL_BAND,
        data["future_growth_change"].abs() <= NEUTRAL_BAND,
    ],
    ["Accelerating", "Decelerating", "Neutral"],
    default="Unavailable",
)

target_summary = (
    data["target_status"]
    .value_counts(dropna=False)
    .rename_axis("target_status")
    .reset_index(name="rows")
)
target_summary["fraction"] = target_summary["rows"] / len(data)
display(target_summary)

## Financial feature engineering

In [ ]:
data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

previous_end = grouped["quarter_end"].shift(1)
gap1 = (data["quarter_end"] - previous_end).dt.days
valid_qoq = gap1.between(60, 125)

quarter_lag4 = grouped["quarter_end"].shift(4)
gap4 = (data["quarter_end"] - quarter_lag4).dt.days
valid_yoy = gap4.between(320, 410)

def add_qoq_change(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(1)
        data[output] = np.where(valid_qoq, data[column] - lagged, np.nan)

def add_yoy_growth(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(4)
        data[output] = np.where(
            valid_yoy & (lagged.abs() > 1e-12),
            data[column] / lagged - 1.0,
            np.nan,
        )

if "revenue" in data.columns:
    revenue_lag1 = grouped["revenue"].shift(1)
    data["sequential_revenue_growth"] = np.where(
        valid_qoq & (revenue_lag1.abs() > 1e-12),
        data["revenue"] / revenue_lag1 - 1.0,
        np.nan,
    )

add_qoq_change("gross_margin", "gross_margin_change")
add_qoq_change("operating_margin", "operating_margin_change")
add_qoq_change("cash_flow_margin", "cash_flow_margin_change")
add_qoq_change(
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
)
add_qoq_change(
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
)
add_qoq_change("capex_to_revenue", "capex_to_revenue_change")

add_yoy_growth("inventory", "inventory_yoy_growth")
add_yoy_growth("accounts_receivable", "receivables_yoy_growth")
add_yoy_growth("capital_expenditures", "capex_yoy_growth")

if "inventory_yoy_growth" in data.columns:
    data["inventory_revenue_growth_gap"] = (
        data["inventory_yoy_growth"] - data["revenue_yoy_growth"]
    )

if "receivables_yoy_growth" in data.columns:
    data["receivables_revenue_growth_gap"] = (
        data["receivables_yoy_growth"] - data["revenue_yoy_growth"]
    )

data["calendar_quarter"] = data["quarter_end"].dt.to_period("Q")

relative_base_features = [
    column
    for column in [
        "revenue_yoy_growth",
        "revenue_momentum",
        "sequential_revenue_growth",
        "gross_margin",
        "gross_margin_change",
        "cash_flow_margin",
        "inventory_to_ttm_revenue",
        "inventory_revenue_growth_gap",
        "receivables_to_ttm_revenue",
        "capex_to_revenue",
    ]
    if column in data.columns
]

quarter_medians = (
    data.groupby("calendar_quarter")[relative_base_features]
    .median()
    .sort_index()
)
prior_quarter_medians = quarter_medians.shift(1).add_suffix(
    "_prior_sector_median"
)

data = data.merge(
    prior_quarter_medians,
    left_on="calendar_quarter",
    right_index=True,
    how="left",
)

for feature in relative_base_features:
    median_column = f"{feature}_prior_sector_median"
    data[f"{feature}_relative_to_sector"] = (
        data[feature] - data[median_column]
    )

engineered_features = [
    column
    for column in data.columns
    if (
        column.endswith("_change")
        or column.endswith("_yoy_growth")
        or column.endswith("_growth_gap")
        or column.endswith("_relative_to_sector")
        or column == "sequential_revenue_growth"
    )
]

print("Engineered features:", len(engineered_features))
display(data[["ticker", "quarter_end"] + engineered_features[:12]].head(10))

## Chronological train/validation/test split

In [ ]:
# ------------------------------------------------------------------
# Chronological split + matched-row lock
# ------------------------------------------------------------------
required_splits = {"train", "validation", "test"}

existing_splits = (
    set(data["split"].dropna().astype(str).str.lower().unique())
    if "split" in data.columns
    else set()
)

existing_split_is_usable = required_splits.issubset(existing_splits)

if existing_split_is_usable:
    data["split"] = data["split"].astype(str).str.lower()
    print("Using the existing chronological train/validation/test split.")
else:
    if "split" in data.columns:
        print(
            "Existing split does not contain train/validation/test; "
            "rebuilding chronologically."
        )

    data["split"] = pd.NA

    if (
        "label_available_date" in data.columns
        and data["label_available_date"].notna().any()
    ):
        split_date_column = "label_available_date"
    elif (
        "feature_cutoff_date" in data.columns
        and data["feature_cutoff_date"].notna().any()
    ):
        split_date_column = "feature_cutoff_date"
    else:
        split_date_column = "quarter_end"

    eligible_dates = (
        data.loc[data["target_clean"].notna(), split_date_column]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(eligible_dates) < 3:
        raise ValueError(
            "At least three distinct dated periods are required."
        )

    train_position = max(
        0,
        min(
            len(eligible_dates) - 3,
            int(len(eligible_dates) * 0.60) - 1,
        ),
    )
    validation_position = max(
        train_position + 1,
        min(
            len(eligible_dates) - 2,
            int(len(eligible_dates) * 0.80) - 1,
        ),
    )

    automatic_train_end = eligible_dates.iloc[train_position]
    automatic_validation_end = eligible_dates.iloc[validation_position]

    labeled = data["target_clean"].notna()
    split_dates = data[split_date_column]

    data.loc[
        labeled & (split_dates <= automatic_train_end),
        "split",
    ] = "train"

    data.loc[
        labeled
        & (split_dates > automatic_train_end)
        & (split_dates <= automatic_validation_end),
        "split",
    ] = "validation"

    data.loc[
        labeled & (split_dates > automatic_validation_end),
        "split",
    ] = "test"

    print("Split date column:", split_date_column)
    print("Automatic train end:", automatic_train_end)
    print("Automatic validation end:", automatic_validation_end)


def make_observation_id(frame: pd.DataFrame) -> pd.Series:
    quarter = pd.to_datetime(
        frame["quarter_end"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    if "cik" in frame.columns:
        company = frame["cik"].astype(str).str.strip()
    elif "ticker" in frame.columns:
        company = frame["ticker"].astype(str).str.strip()
    else:
        raise ValueError("Need cik or ticker to create observation_id.")

    return company + "__" + quarter.fillna("missing_date")


data["observation_id"] = make_observation_id(data)

# First form the labeled chronological universe.
model_data = data[
    data["target_clean"].notna()
    & data["split"].isin(["train", "validation", "test"])
].copy()

model_data["target_clean"] = model_data["target_clean"].astype(int)

# ------------------------------------------------------------------
# Prefer the exact Branch 3 v7 manifest if it exists.
# ------------------------------------------------------------------
def resolve_equal_row_manifest():
    target_name = "master_equal_row_manifest.csv"

    for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if root.exists():
            matches = list(root.rglob(target_name))
            if matches:
                return matches[0]

    candidates = [
        Path("/content/master_equal_row_manifest.csv"),
        Path("/content/semiconductor_branch_combined/master_equal_row_manifest.csv"),
        Path("/content/drive/MyDrive/master_equal_row_manifest.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


manifest_path = resolve_equal_row_manifest()

if manifest_path is not None:
    manifest = pd.read_csv(manifest_path)

    required_manifest_columns = {
        "observation_id",
        "split",
    }
    missing = required_manifest_columns - set(manifest.columns)
    if missing:
        raise ValueError(
            f"Matched-row manifest is missing columns: {sorted(missing)}"
        )

    manifest["split"] = manifest["split"].astype(str).str.lower()
    manifest_ids = set(manifest["observation_id"].astype(str))

    model_data = model_data[
        model_data["observation_id"].astype(str).isin(manifest_ids)
    ].copy()

    expected_split = (
        manifest[["observation_id", "split"]]
        .drop_duplicates("observation_id")
        .set_index("observation_id")["split"]
    )

    model_data["split"] = (
        model_data["observation_id"]
        .map(expected_split)
        .astype(str)
    )

    print("Using exact Branch 3 matched-row manifest:")
    print(manifest_path)

else:
    # Fallback: independently reproduce Branch 3's usable-text criterion.
    # This works when the multimodal input already contains the filing text.
    TEXT_COLUMN_CANDIDATES = [
        "sec_text",
        "filing_text",
        "document_text",
        "mda_text",
        "management_discussion_text",
        "risk_factors_text",
        "risk_text",
    ]
    available_text_columns = [
        c for c in TEXT_COLUMN_CANDIDATES
        if c in model_data.columns
    ]

    if not available_text_columns:
        raise FileNotFoundError(
            "No master_equal_row_manifest.csv was found and the input dataset "
            "contains no SEC text columns. Run Branch 3 v7 first so it writes "
            "master_equal_row_manifest.csv, then rerun Branch 1."
        )

    sec_text_for_matching = (
        model_data[available_text_columns]
        .fillna("")
        .astype(str)
        .apply(
            lambda row: "\n\n".join(
                value.strip()
                for value in row
                if value and value.strip()
            ),
            axis=1,
        )
    )

    # Same minimum usable-text rule as Branch 3.
    MIN_TEXT_CHARS_FOR_MATCHING = 500
    has_sec_text_for_matching = (
        sec_text_for_matching.str.len()
        >= MIN_TEXT_CHARS_FOR_MATCHING
    )

    # If filing timing columns exist, exclude text that would not have been
    # available at the feature cutoff, matching the combined-branch logic.
    if {
        "filing_date",
        "feature_cutoff_date",
    }.issubset(model_data.columns):
        filing_date = pd.to_datetime(
            model_data["filing_date"], errors="coerce"
        )
        cutoff_date = pd.to_datetime(
            model_data["feature_cutoff_date"], errors="coerce"
        )
        late_text = (
            filing_date.notna()
            & cutoff_date.notna()
            & (filing_date > cutoff_date)
        )
        has_sec_text_for_matching = (
            has_sec_text_for_matching & ~late_text
        )

    model_data = model_data[
        has_sec_text_for_matching
    ].copy()

    print(
        "No Branch 3 manifest found; derived matched rows locally "
        "using the same >=500-character SEC-text rule."
    )


# ------------------------------------------------------------------
# Integrity checks and paper-facing manifest
# ------------------------------------------------------------------
if model_data["observation_id"].duplicated().any():
    duplicates = int(
        model_data["observation_id"].duplicated().sum()
    )
    raise ValueError(
        f"Found {duplicates} duplicate matched observation IDs."
    )

available_splits = set(model_data["split"].dropna().unique())
missing_splits = required_splits - available_splits
if missing_splits:
    raise ValueError(
        f"Matched data are missing splits: {sorted(missing_splits)}."
    )

for split_name in ["train", "validation", "test"]:
    split_frame = model_data[
        model_data["split"] == split_name
    ]
    if split_frame["target_clean"].nunique() < 2:
        raise ValueError(
            f"{split_name} matched rows contain only one target class."
        )

split_summary = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        first_quarter=("quarter_end", "min"),
        last_quarter=("quarter_end", "max"),
        acceleration_rate=("target_clean", "mean"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

display(split_summary)

MATCHED_ROW_MANIFEST = (
    model_data[
        [
            c for c in [
                "observation_id",
                "split",
                "cik",
                "ticker",
                "company_name",
                "quarter_end",
                "target_clean",
            ]
            if c in model_data.columns
        ]
    ]
    .sort_values(["split", "observation_id"])
    .reset_index(drop=True)
)

print(
    "Matched rows locked. Train / validation / test:",
    *[
        int((model_data["split"] == s).sum())
        for s in ["train", "validation", "test"]
    ],
)


## Numerical feature list

In [ ]:
candidate_numeric_features = [
    'revenue_yoy_growth', 'revenue_momentum', 'sequential_revenue_growth',
    'gross_margin', 'gross_margin_change', 'operating_margin',
    'operating_margin_change', 'cash_flow_margin', 'cash_flow_margin_change',
    'inventory_to_ttm_revenue', 'inventory_to_ttm_revenue_change',
    'inventory_yoy_growth', 'inventory_revenue_growth_gap',
    'receivables_to_ttm_revenue', 'receivables_to_ttm_revenue_change',
    'receivables_yoy_growth', 'receivables_revenue_growth_gap',
    'capex_to_revenue', 'capex_to_revenue_change', 'capex_yoy_growth',
    'log_assets', 'liabilities_to_assets',
]
candidate_numeric_features += [
    f'{feature}_relative_to_sector' for feature in relative_base_features
]

training_feature_frame = model_data.loc[
    model_data["split"].eq("train")
].copy()

numeric_features = [
    c for c in dict.fromkeys(candidate_numeric_features)
    if c in training_feature_frame.columns
    and training_feature_frame[c].notna().sum() >= 10
    and training_feature_frame[c].nunique(dropna=True) > 1
]
categorical_features = [
    c for c in ['fiscal_quarter', 'semiconductor_subgroup']
    if c in training_feature_frame.columns
    and training_feature_frame[c].notna().any()
]
feature_columns = numeric_features + categorical_features
print('Numerical features selected from TRAIN only:', len(numeric_features))
print('Categorical features:', categorical_features)
display(pd.DataFrame({'feature': feature_columns}))


## Train and compare all numerical models

## Information-availability audit


In [ ]:
availability_audit = []
for split_name in ["train", "validation", "test"]:
    part = model_data.loc[model_data["split"].eq(split_name)].copy()
    row = {"split": split_name, "rows": len(part)}

    filing_col = next(
        (c for c in ["source_filing_date", "filing_date"] if c in part.columns),
        None,
    )
    cutoff_col = "feature_cutoff_date" if "feature_cutoff_date" in part.columns else None

    if filing_col and cutoff_col:
        filing = pd.to_datetime(part[filing_col], errors="coerce")
        cutoff = pd.to_datetime(part[cutoff_col], errors="coerce")
        checked = filing.notna() & cutoff.notna()
        row["filing_timing_rows_checked"] = int(checked.sum())
        row["filings_after_feature_cutoff"] = int(
            (checked & (filing > cutoff)).sum()
        )
    else:
        row["filing_timing_rows_checked"] = 0
        row["filings_after_feature_cutoff"] = np.nan

    if "label_available_date" in part.columns and cutoff_col:
        label_date = pd.to_datetime(part["label_available_date"], errors="coerce")
        cutoff = pd.to_datetime(part[cutoff_col], errors="coerce")
        checked = label_date.notna() & cutoff.notna()
        row["label_timing_rows_checked"] = int(checked.sum())
        row["labels_available_by_same_row_cutoff"] = int(
            (checked & (label_date <= cutoff)).sum()
        )
    else:
        row["label_timing_rows_checked"] = 0
        row["labels_available_by_same_row_cutoff"] = np.nan

    availability_audit.append(row)

availability_audit = pd.DataFrame(availability_audit)
display(availability_audit)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Kaggle + Colab output location.
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path(
        "/kaggle/working/datasets/pjbob1/"
        "semiconductor_branch_numerical_v4_paper_ready"
    )
else:
    OUTPUT_DIR = Path(
        "/content/semiconductor_branch_numerical_v4_paper_ready"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the exact numerical comparison rows.
MATCHED_ROW_MANIFEST.to_csv(
    OUTPUT_DIR / "master_equal_row_manifest_branch1.csv",
    index=False,
)

RUN_MLP = False


def make_preprocessor():
    transformers = []

    if numeric_features:
        transformers.append((
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ))

    if categorical_features:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical_features,
        ))

    return ColumnTransformer(
        transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


def choose_threshold(
    y_true,
    probabilities,
    threshold_grid=None,
):
    """
    Pick the validation threshold maximizing balanced accuracy.
    If multiple thresholds tie, prefer the one closest to 0.50.
    """
    if threshold_grid is None:
        threshold_grid = THRESHOLD_GRID

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    if len(y_true) != len(probabilities):
        raise ValueError(
            "Threshold tuning requires equal-length labels/probabilities."
        )
    if len(np.unique(y_true)) < 2:
        raise ValueError(
            "Validation threshold tuning requires both classes."
        )
    if not np.isfinite(probabilities).all():
        raise ValueError(
            "Validation probabilities contain NaN/inf."
        )

    scored = []

    for threshold in threshold_grid:
        pred = (
            probabilities >= threshold
        ).astype(int)

        scored.append((
            float(threshold),
            float(
                balanced_accuracy_score(
                    y_true,
                    pred,
                )
            ),
        ))

    best_score = max(
        score for _, score in scored
    )

    tied = [
        item
        for item in scored
        if np.isclose(
            item[1],
            best_score,
            rtol=0.0,
            atol=1e-12,
        )
    ]

    return min(
        tied,
        key=lambda item: abs(item[0] - 0.50),
    )


def evaluate_row(
    model_name,
    y_true,
    probabilities,
    predictions,
    baseline_probability,
):
    row = {
        "model": model_name,
        "rows": len(y_true),
        "accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            predictions,
        ),
        "macro_f1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "acceleration_precision": precision_score(
            y_true,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "acceleration_recall": recall_score(
            y_true,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "deceleration_recall": recall_score(
            y_true,
            predictions,
            pos_label=0,
            zero_division=0,
        ),
        "brier_score": brier_score_loss(
            y_true,
            probabilities,
        ),
        "log_loss": log_loss(
            y_true,
            np.c_[
                1 - probabilities,
                probabilities,
            ],
            labels=[0, 1],
        ),
    }

    base = np.full(
        len(y_true),
        baseline_probability,
    )

    base_brier = brier_score_loss(
        y_true,
        base,
    )

    row["brier_skill_score"] = (
        1 - row["brier_score"] / base_brier
        if base_brier > 0
        else np.nan
    )

    row["roc_auc"] = (
        roc_auc_score(
            y_true,
            probabilities,
        )
        if pd.Series(y_true).nunique() == 2
        else np.nan
    )

    row["average_precision"] = (
        average_precision_score(
            y_true,
            probabilities,
        )
        if pd.Series(y_true).nunique() == 2
        else np.nan
    )

    return row


def candidate_is_better(candidate, best):
    # PRIMARY: validation balanced accuracy.
    if best is None:
        return True

    if (
        candidate["validation_balanced_accuracy"]
        > best["validation_balanced_accuracy"] + 1e-12
    ):
        return True

    # SECONDARY: lower validation Brier score.
    if np.isclose(
        candidate["validation_balanced_accuracy"],
        best["validation_balanced_accuracy"],
        rtol=0.0,
        atol=1e-12,
    ):
        if (
            candidate["validation_brier"]
            < best["validation_brier"] - 1e-12
        ):
            return True

    return False


def xgb_compute_kwargs():
    # Use GPU for XGBoost on Kaggle when supported.
    try:
        import torch
        if torch.cuda.is_available():
            return {
                "tree_method": "hist",
                "device": "cuda",
            }
    except Exception:
        pass

    return {
        "tree_method": "hist",
    }


train = model_data[
    model_data["split"] == "train"
].copy()

validation = model_data[
    model_data["split"] == "validation"
].copy()

test = model_data[
    model_data["split"] == "test"
].copy()

y_train = train["target_clean"].astype(int)
y_val = validation["target_clean"].astype(int)
y_test = test["target_clean"].astype(int)

# Baseline comes from TRAIN ONLY.
baseline_probability = float(
    y_train.mean()
)

print(
    "Matched model rows:",
    "train=", len(train),
    "validation=", len(validation),
    "test=", len(test),
)

# Verify exact row lock one final time.
for split_name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    expected = set(
        MATCHED_ROW_MANIFEST.loc[
            MATCHED_ROW_MANIFEST["split"].eq(split_name),
            "observation_id",
        ]
    )
    actual = set(
        frame["observation_id"]
    )

    if actual != expected:
        raise AssertionError(
            f"{split_name} rows differ from matched-row manifest."
        )

print("Equal-row assertions passed.")


# ------------------------------------------------------------------
# Candidate model families
# ------------------------------------------------------------------
model_candidates = []

for C in C_GRID:
    model_candidates.append((
        "L2 Logistic Regression",
        {
            "C": C,
            "class_weight": None,
        },
    ))

    model_candidates.append((
        "Balanced L2 Logistic Regression",
        {
            "C": C,
            "class_weight": "balanced",
        },
    ))

for params in [
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 500,
        "max_depth": 10,
        "min_samples_leaf": 3,
        "max_features": 0.7,
    },
]:
    model_candidates.append((
        "Random Forest",
        params,
    ))

positive = max(
    1,
    int((y_train == 1).sum()),
)
negative = max(
    1,
    int((y_train == 0).sum()),
)
scale_pos_weight = negative / positive

for params in [
    {
        "n_estimators": 200,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
    {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
]:
    model_candidates.append((
        "XGBoost",
        {
            **params,
            "scale_pos_weight": scale_pos_weight,
        },
    ))

if RUN_MLP:
    for params in [
        {
            "hidden_layer_sizes": (32,),
            "alpha": 0.001,
        },
        {
            "hidden_layer_sizes": (64, 32),
            "alpha": 0.001,
        },
        {
            "hidden_layer_sizes": (64, 32),
            "alpha": 0.01,
        },
    ]:
        model_candidates.append((
            "MLP Neural Network",
            params,
        ))


def build_model(
    name,
    params,
):
    if name in {
        "L2 Logistic Regression",
        "Balanced L2 Logistic Regression",
    }:
        classifier = LogisticRegression(
            C=params["C"],
            penalty="l2",
            class_weight=params["class_weight"],
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )

    elif name == "Random Forest":
        classifier = RandomForestClassifier(
            **params,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    elif name == "XGBoost":
        classifier = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **xgb_compute_kwargs(),
        )

    else:
        classifier = MLPClassifier(
            **params,
            activation="relu",
            solver="adam",
            max_iter=1500,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=RANDOM_STATE,
        )

    return Pipeline([
        (
            "preprocessor",
            make_preprocessor(),
        ),
        (
            "classifier",
            classifier,
        ),
    ])


# ------------------------------------------------------------------
# Validation-only model and threshold selection
# ------------------------------------------------------------------
best_by_model = {}
selection_rows = []

for candidate_number, (
    name,
    params,
) in enumerate(
    model_candidates,
    start=1,
):
    print(
        f"[{candidate_number}/{len(model_candidates)}]",
        name,
        params,
    )

    pipe = build_model(
        name,
        params,
    )

    pipe.fit(
        train[feature_columns],
        y_train,
    )

    validation_probability = (
        pipe.predict_proba(
            validation[feature_columns]
        )[:, 1]
    )

    threshold, val_bal = choose_threshold(
        y_val,
        validation_probability,
    )

    validation_brier = brier_score_loss(
        y_val,
        validation_probability,
    )

    candidate = {
        "params": params,
        "threshold": float(threshold),
        "validation_balanced_accuracy": float(val_bal),
        "validation_brier": float(validation_brier),
    }

    selection_rows.append({
        "model": name,
        "params": str(params),
        "validation_balanced_accuracy": val_bal,
        "validation_brier": validation_brier,
        "threshold": threshold,
    })

    if candidate_is_better(
        candidate,
        best_by_model.get(name),
    ):
        best_by_model[name] = candidate


# ------------------------------------------------------------------
# Test evaluation — no test-set tuning
# ------------------------------------------------------------------
results = []

prediction_id_columns = [
    c
    for c in [
        "observation_id",
        "cik",
        "ticker",
        "company_name",
        "quarter_end",
        "future_growth_change",
        "target_clean",
    ]
    if c in test.columns
]

predictions = test[
    prediction_id_columns
].copy()

final_models = {}

# Baselines.
prior_probability = np.full(
    len(test),
    baseline_probability,
)

prior_prediction = (
    prior_probability >= 0.5
).astype(int)

results.append(
    evaluate_row(
        "Prior-probability baseline",
        y_test,
        prior_probability,
        prior_prediction,
        baseline_probability,
    )
)

majority_class = int(
    y_train.mode().iloc[0]
)

majority_probability = np.full(
    len(test),
    float(majority_class),
)

majority_prediction = np.full(
    len(test),
    majority_class,
)

results.append(
    evaluate_row(
        "Majority-class baseline",
        y_test,
        majority_probability,
        majority_prediction,
        baseline_probability,
    )
)


for name, chosen in best_by_model.items():
    params = chosen["params"]

    # IMPORTANT:
    # Fit on TRAIN ONLY because threshold was calibrated on predictions from
    # a TRAIN-fitted model. This keeps model probability distribution and
    # threshold calibration aligned.
    final_model = build_model(
        name,
        params,
    )

    final_model.fit(
        train[feature_columns],
        y_train,
    )

    test_probability = (
        final_model.predict_proba(
            test[feature_columns]
        )[:, 1]
    )

    test_prediction = (
        test_probability
        >= chosen["threshold"]
    ).astype(int)

    result_row = evaluate_row(
        name,
        y_test,
        test_probability,
        test_prediction,
        baseline_probability,
    )

    result_row.update({
        "selected_params": str(params),
        "threshold": chosen["threshold"],
        "validation_balanced_accuracy": (
            chosen[
                "validation_balanced_accuracy"
            ]
        ),
        "validation_brier": (
            chosen["validation_brier"]
        ),
        "selection_metric": (
            "validation_balanced_accuracy"
        ),
    })

    results.append(
        result_row
    )

    predictions[
        f"{name}_probability"
    ] = test_probability

    predictions[
        f"{name}_prediction"
    ] = test_prediction

    final_models[name] = final_model


selection_table = (
    pd.DataFrame(selection_rows)
    .sort_values(
        [
            "model",
            "validation_balanced_accuracy",
            "validation_brier",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

results_table = (
    pd.DataFrame(results)
    .sort_values(
        [
            "balanced_accuracy",
            "brier_score",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(results_table)
display(selection_table)

# Paper-integrity check: every result row must show the same test N.
model_test_counts = (
    results_table[
        ~results_table["model"].str.contains(
            "baseline",
            case=False,
            regex=False,
        )
    ]["rows"]
    .unique()
)

if len(model_test_counts) != 1:
    raise AssertionError(
        "Numerical models were not evaluated on one identical test set."
    )

print(
    "All numerical models evaluated on N test =",
    int(model_test_counts[0]),
)

selection_table.to_csv(
    OUTPUT_DIR / "numerical_validation_selection.csv",
    index=False,
)

results_table.to_csv(
    OUTPUT_DIR / "numerical_test_results.csv",
    index=False,
)

predictions.to_csv(
    OUTPUT_DIR / "numerical_test_predictions.csv",
    index=False,
)

split_summary.to_csv(
    OUTPUT_DIR / "matched_split_summary.csv",
    index=False,
)

with pd.ExcelWriter(
    OUTPUT_DIR / "numerical_branch_results.xlsx",
    engine="openpyxl",
) as writer:
    results_table.to_excel(
        writer,
        sheet_name="Model_Results",
        index=False,
    )
    selection_table.to_excel(
        writer,
        sheet_name="Validation_Selection",
        index=False,
    )
    predictions.to_excel(
        writer,
        sheet_name="Test_Predictions",
        index=False,
    )
    MATCHED_ROW_MANIFEST.to_excel(
        writer,
        sheet_name="Equal_Row_Manifest",
        index=False,
    )
    split_summary.to_excel(
        writer,
        sheet_name="Split_Summary",
        index=False,
    )

joblib.dump(
    final_models,
    OUTPUT_DIR / "numerical_final_models.joblib",
)

print("Saved to:", OUTPUT_DIR)


## Secondary numerical ablation study

This section is optional robustness analysis. The main paper comparison is the matched-row model table above.


In [ ]:
feature_groups = {
    "revenue": [c for c in feature_columns if any(k in c for k in [
        "revenue_yoy_growth", "revenue_momentum", "sequential_revenue_growth"
    ])],
    "profitability": [c for c in feature_columns if "gross_margin" in c or "operating_margin" in c],
    "cash_flow": [c for c in feature_columns if "cash_flow_margin" in c],
    "inventory": [c for c in feature_columns if "inventory" in c],
    "receivables": [c for c in feature_columns if "receivable" in c or "ar_" in c],
    "capex": [c for c in feature_columns if "capex" in c],
    "balance_sheet": [c for c in feature_columns if "log_assets" in c or "liabilities_to_assets" in c],
}

non_baseline = results_table[
    ~results_table["model"].str.contains("baseline", case=False, regex=False)
].copy()

best_main_name = non_baseline.sort_values(
    ["validation_balanced_accuracy", "validation_brier"],
    ascending=[False, True],
).iloc[0]["model"]
best_main_spec = best_by_model[best_main_name]

print("Numerical ablation model family:", best_main_name)

def build_feature_subset_model(model_name, params, selected_features):
    num = [c for c in selected_features if c in numeric_features]
    cat = [c for c in selected_features if c in categorical_features]

    transformers = []
    if num:
        transformers.append((
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num,
        ))
    if cat:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat,
        ))

    preprocessor = ColumnTransformer(
        transformers, remainder="drop", verbose_feature_names_out=False
    )

    if model_name in {"L2 Logistic Regression", "Balanced L2 Logistic Regression"}:
        clf = LogisticRegression(
            C=params["C"], penalty="l2",
            class_weight=params["class_weight"],
            solver="lbfgs", max_iter=5000, random_state=RANDOM_STATE,
        )
    elif model_name == "Random Forest":
        clf = RandomForestClassifier(
            **params, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    elif model_name == "XGBoost":
        clf = XGBClassifier(
            **params, objective="binary:logistic", eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1, **xgb_compute_kwargs(),
        )
    else:
        raise ValueError(model_name)

    return Pipeline([("preprocessor", preprocessor), ("classifier", clf)])

def run_feature_ablation(label, selected_features):
    model = build_feature_subset_model(
        best_main_name, best_main_spec["params"], selected_features
    )
    model.fit(train[selected_features], y_train)
    val_p = model.predict_proba(validation[selected_features])[:, 1]
    threshold, val_ba = choose_threshold(y_val, val_p)
    val_brier = brier_score_loss(y_val, val_p)

    test_p = model.predict_proba(test[selected_features])[:, 1]
    test_pred = (test_p >= threshold).astype(int)

    row = evaluate_row(
        label, y_test, test_p, test_pred, baseline_probability
    )
    row.update({
        "validation_balanced_accuracy": val_ba,
        "validation_brier": val_brier,
        "selected_threshold": threshold,
        "selected_model_family": best_main_name,
        "feature_count": len(selected_features),
    })
    return row

ablation_rows = [run_feature_ablation("Full numerical feature set", feature_columns)]
for group_name, group_columns in feature_groups.items():
    remaining = [c for c in feature_columns if c not in set(group_columns)]
    if not group_columns or not remaining:
        continue
    row = run_feature_ablation(f"Without {group_name}", remaining)
    row["removed_group"] = group_name
    row["removed_feature_count"] = len(group_columns)
    ablation_rows.append(row)

numerical_ablation_results = pd.DataFrame(ablation_rows)
full_ba = float(
    numerical_ablation_results.loc[
        numerical_ablation_results["model"].eq("Full numerical feature set"),
        "balanced_accuracy",
    ].iloc[0]
)
numerical_ablation_results["balanced_accuracy_delta_vs_full"] = (
    numerical_ablation_results["balanced_accuracy"] - full_ba
)
display(numerical_ablation_results.sort_values(
    "balanced_accuracy_delta_vs_full", ascending=False
))
numerical_ablation_results.to_csv(
    OUTPUT_DIR/"numerical_feature_ablation_v4.csv", index=False
)


## Robustness — unseen-company generalization

Leave one company out entirely, train on the remaining companies, and evaluate
on the unseen firm. This is reported separately from the primary chronological
test.

In [ ]:
RUN_LOCO = False

if not RUN_LOCO:
    print("Chronological LOCO skipped. Set RUN_LOCO=True for robustness analysis.")
else:
    company_col = "cik" if "cik" in model_data.columns else "ticker"
    time_col = "feature_cutoff_date" if "feature_cutoff_date" in model_data.columns else "quarter_end"
    chronological_loco_rows = []

    for company_value in sorted(model_data[company_col].dropna().unique()):
        held_out = model_data[
            model_data[company_col].eq(company_value)
            & model_data["split"].eq("test")
        ].copy()
        if held_out.empty:
            continue

        first_time = pd.to_datetime(
            held_out[time_col], errors="coerce"
        ).min()

        training_pool = model_data[
            ~model_data[company_col].eq(company_value)
        ].copy()
        training_pool = training_pool[
            pd.to_datetime(training_pool[time_col], errors="coerce") < first_time
        ]

        if len(held_out) < 2 or training_pool["target_clean"].nunique() < 2:
            continue

        model = build_feature_subset_model(
            best_main_name, best_main_spec["params"], feature_columns
        )
        model.fit(
            training_pool[feature_columns],
            training_pool["target_clean"].astype(int),
        )
        p = model.predict_proba(held_out[feature_columns])[:, 1]
        pred = (p >= best_main_spec["threshold"]).astype(int)

        row = evaluate_row(
            f"Chronological LOCO {company_value}",
            held_out["target_clean"].astype(int),
            p, pred, float(training_pool["target_clean"].mean()),
        )
        row[company_col] = company_value
        row["n_test_rows"] = len(held_out)
        row["training_rows"] = len(training_pool)
        chronological_loco_rows.append(row)

    chronological_loco_results = pd.DataFrame(chronological_loco_rows)
    display(chronological_loco_results)
    chronological_loco_results.to_csv(
        OUTPUT_DIR/"chronological_loco_v4.csv", index=False
    )
